# 第 7 课：N-best、Lattice、分数融合与二遍重打分

前 6 课回答了“语言模型怎样变成图”。这一课回答“图进入解码后，怎样和声学分一起决定结果”。

完成后你应该能够：

1. 区分 1-best、N-best 与 lattice；
2. 解释声学代价、LM 代价、LM scale、插词项和热词奖励的方向；
3. 用 OpenFst 从迷你 lattice 提取唯一 N-best；
4. 计算 top-k oracle WER，判断正确候选是否还活着；
5. 实现一个只重排候选、不凭空生成文本的二遍重打分器。


## 0. 三个对象不要混在一起

| 对象 | 保存什么 | 优点 | 损失 |
|---|---|---|---|
| 1-best | 最好的一条完整路径 | 最小、最简单 | 其他候选全部丢失 |
| N-best | N 条完整候选 | 方便二遍模型批量打分 | 重复保存公共前后缀 |
| lattice | 共享状态和弧的候选图 | 紧凑，保留更多竞争关系 | 操作比列表复杂 |

生产 lattice 还常带帧时间、声学对齐和独立分数组件。本课先用共享词前缀的可见小图掌握核心关系。


In [1]:
from pathlib import Path
from math import log
import subprocess
import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display

def find_project_root():
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / 'pyproject.toml').exists():
            return candidate
    raise FileNotFoundError('请从 learn_asr 项目或 notebooks 目录启动 Jupyter')

ROOT = find_project_root()
PREV = ROOT / 'openfst_lab' / 'lesson05'
LAB = ROOT / 'openfst_lab' / 'lesson07'
LAB.mkdir(parents=True, exist_ok=True)
words_path = PREV / 'words.txt'
arpa_path = PREV / 'tiny.3gram.arpa'
if not words_path.exists() or not arpa_path.exists():
    raise FileNotFoundError('请先完整运行第 5 课，生成 words.txt 与 tiny.3gram.arpa')

def run_wsl(*args, check=True):
    result = subprocess.run(
        ['wsl', '-d', 'Ubuntu', '--', *map(str, args)],
        text=True, capture_output=True, check=False, encoding='utf-8', errors='replace',
    )
    if check and result.returncode != 0:
        raise RuntimeError(f'命令失败：{args}\nstdout:\n{result.stdout}\nstderr:\n{result.stderr}')
    return result

def to_wsl_path(path):
    resolved = Path(path).resolve()
    drive = resolved.drive.rstrip(':').lower()
    relative = resolved.relative_to(resolved.anchor).as_posix()
    return f'/mnt/{drive}/{relative}'

def write_lf(path, text):
    Path(path).write_text(text, encoding='utf-8', newline='\n')

for command in ['fstcompile', 'fstinfo', 'fstprint', 'fstshortestpath']:
    location = run_wsl('which', command).stdout.strip()
    print(f'{command:18s}', location or 'MISSING')
    assert location


fstcompile         /usr/bin/fstcompile
fstinfo            /usr/bin/fstinfo


fstprint           /usr/bin/fstprint
fstshortestpath    /usr/bin/fstshortestpath


## 1. 从 ARPA 取得每个候选的 LM 代价

仍使用上一课验证过的换算：`LM cost = -log10(P) × ln(10)`。候选的声学代价是假设一遍声学模型产生的教学数据；数值越低代表越符合音频。


In [2]:
def parse_arpa(path):
    entries = {}
    order = 0
    for raw in Path(path).read_text(encoding='utf-8').splitlines():
        line = raw.strip()
        if line.startswith('\\') and line.endswith('-grams:'):
            order = int(line[1:].split('-')[0])
            entries[order] = {}
        elif order and line and not line.startswith('\\'):
            fields = line.split()
            probability = float(fields[0])
            ngram = tuple(fields[1:1 + order])
            backoff = float(fields[1 + order]) if len(fields) > 1 + order else 0.0
            entries[order][ngram] = (probability, backoff)
    return entries

arpa = parse_arpa(arpa_path)
max_order = max(arpa)

def arpa_word_log10(word, history):
    accumulated_backoff = 0.0
    max_history = min(len(history), max_order - 1)
    for history_length in range(max_history, -1, -1):
        current_history = tuple(history[-history_length:]) if history_length else ()
        candidate = current_history + (word,)
        if candidate in arpa[len(candidate)]:
            return accumulated_backoff + arpa[len(candidate)][candidate][0]
        if current_history:
            accumulated_backoff += arpa[len(current_history)].get(current_history, (0.0, 0.0))[1]
    raise KeyError(word)

def sentence_lm_cost(text):
    tokens = ['<s>', *text.split(), '</s>']
    total_log10 = sum(arpa_word_log10(tokens[i], tokens[:i]) for i in range(1, len(tokens)))
    return -total_log10 * log(10)

CANDIDATES = [
    {'text': 'jintian tianqi henhao',       'acoustic': 4.8},
    {'text': 'jintian tianqi bucuo',        'acoustic': 4.4},
    {'text': 'jintian xinqing henhao',      'acoustic': 3.5},
    {'text': 'jintian xinqing bucuo',       'acoustic': 3.2},
    {'text': 'mingtian tianqi henhao',      'acoustic': 2.6},
    {'text': 'zuotian tianqi bucuo',        'acoustic': 2.8},
    {'text': 'jintian tianqi',               'acoustic': 3.8},
    {'text': 'jintian tianqi hen leng',      'acoustic': 3.0},
]
for candidate in CANDIDATES:
    candidate['lm'] = sentence_lm_cost(candidate['text'])
    candidate['words'] = len(candidate['text'].split())

print(f'{"候选":32s} {"声学代价":>8s} {"LM代价":>8s}')
for candidate in CANDIDATES:
    print(f'{candidate["text"]:32s} {candidate["acoustic"]:8.3f} {candidate["lm"]:8.3f}')


候选                                   声学代价     LM代价
jintian tianqi henhao               4.800    4.553
jintian tianqi bucuo                4.400    4.354
jintian xinqing henhao              3.500    4.715
jintian xinqing bucuo               3.200    4.715
mingtian tianqi henhao              2.600    5.342
zuotian tianqi bucuo                2.800    5.337
jintian tianqi                      3.800    5.557
jintian tianqi hen leng             3.000    5.816


## 2. 分数公式与符号方向

本课统一使用 **cost/代价域，越低越好**：

$$C = C_{acoustic} + \alpha C_{LM} + \beta N_{word} - \gamma N_{hotword}$$

- `α`：LM scale；越大越相信语言模型；
- `β > 0`：每多一个词都增加代价，倾向短句；`β < 0` 则奖励更多词；
- `γ > 0`：从代价中减去热词奖励；
- 不同工具可能输出越大越好的 score。调参前必须先确认符号约定。


In [3]:
def rank_candidates(lm_scale=1.0, insertion_penalty=0.0, hotword='', hotword_bonus=0.0):
    ranked = []
    for candidate in CANDIDATES:
        hot_count = candidate['text'].split().count(hotword) if hotword else 0
        total = (candidate['acoustic'] + lm_scale * candidate['lm']
                 + insertion_penalty * candidate['words'] - hotword_bonus * hot_count)
        ranked.append({**candidate, 'hot_count': hot_count, 'total': total})
    return sorted(ranked, key=lambda item: item['total'])

for lm_scale in [0.0, 0.5, 1.0, 1.5]:
    best = rank_candidates(lm_scale=lm_scale)[0]
    print(f'LM scale={lm_scale:3.1f} → {best["text"]:30s} total={best["total"]:.3f}')
assert rank_candidates(lm_scale=0.0)[0]['text'] == 'mingtian tianqi henhao'
assert rank_candidates(lm_scale=1.0)[0]['text'] == 'jintian xinqing bucuo'


LM scale=0.0 → mingtian tianqi henhao         total=2.600
LM scale=0.5 → mingtian tianqi henhao         total=5.271
LM scale=1.0 → jintian xinqing bucuo          total=7.915
LM scale=1.5 → jintian xinqing bucuo          total=10.273


## 3. 把候选压成共享前缀的 lattice

每条候选都单独保存需要 24 个 word arc。用 trie 共享相同前缀后，只需保存一次 `jintian`、`jintian tianqi` 等公共部分。为便于观察，本实验把整条路径的融合代价放在终止权重上。


In [4]:
def build_lattice_text(ranked):
    transitions = {}
    finals = {}
    next_state = 1
    for candidate in ranked:
        state = 0
        for word in candidate['text'].split():
            key = (state, word)
            if key not in transitions:
                transitions[key] = next_state
                next_state += 1
            state = transitions[key]
        finals[state] = candidate['total']
    lines = [f'{source} {destination} {word} 0'
             for (source, word), destination in sorted(transitions.items())]
    lines.extend(f'{state} {weight:.9f}' for state, weight in sorted(finals.items()))
    return '\n'.join(lines) + '\n', len(transitions), next_state

base_ranked = rank_candidates(lm_scale=1.0)
lattice_text, lattice_arcs, lattice_states = build_lattice_text(base_ranked)
lattice_txt_path = LAB / 'tiny_lattice.txt'
lattice_fst = LAB / 'tiny_lattice.fst'
write_lf(lattice_txt_path, lattice_text)
run_wsl(
    'fstcompile', '--acceptor=true',
    f'--isymbols={to_wsl_path(words_path)}', f'--osymbols={to_wsl_path(words_path)}',
    '--keep_isymbols=true', '--keep_osymbols=true',
    to_wsl_path(lattice_txt_path), to_wsl_path(lattice_fst),
)
raw_arc_count = sum(candidate['words'] for candidate in CANDIDATES)
print(f'分别保存所有候选：{raw_arc_count} 条 word arc')
print(f'共享前缀 lattice：{lattice_arcs} 条 word arc，{lattice_states} 个状态')
print('\nOpenFst 检查：')
for line in run_wsl('fstinfo', to_wsl_path(lattice_fst)).stdout.splitlines():
    if line.strip().startswith(('# of states', '# of arcs', '# of final states', 'acyclic', 'acceptor')):
        print(line.strip())
assert lattice_arcs < raw_arc_count


分别保存所有候选：24 条 word arc
共享前缀 lattice：15 条 word arc，16 个状态

OpenFst 检查：


# of states                                       16
# of arcs                                         15
# of final states                                 8
acceptor                                          y


## 4. 用 OpenFst 提取唯一 N-best

`fstshortestpath --nshortest=N --unique=true` 返回 N 条不同标签串。输出仍然是一张 FST，不是已经排好版的 Python 列表，因此下面沿图枚举完整路径并按总代价排序。


In [5]:
def parse_acceptor_paths(fst_path):
    printed = run_wsl('fstprint', '--acceptor=true', to_wsl_path(fst_path)).stdout
    info = run_wsl('fstinfo', to_wsl_path(fst_path)).stdout
    initial = int(next(line.split()[-1] for line in info.splitlines()
                       if line.strip().startswith('initial state')))
    arcs = {}
    finals = {}
    for line in printed.splitlines():
        fields = line.split()
        if len(fields) >= 3:
            source, destination, label = int(fields[0]), int(fields[1]), fields[2]
            weight = float(fields[3]) if len(fields) >= 4 else 0.0
            arcs.setdefault(source, []).append((destination, label, weight))
        elif len(fields) in {1, 2}:
            finals[int(fields[0])] = float(fields[1]) if len(fields) == 2 else 0.0
    paths = []
    def visit(state, labels, cost):
        if state in finals:
            paths.append({'text': ' '.join(labels), 'cost': cost + finals[state]})
        for destination, label, weight in arcs.get(state, []):
            visit(destination, labels + ([] if label == '<eps>' else [label]), cost + weight)
    visit(initial, [], 0.0)
    return sorted(paths, key=lambda item: item['cost'])

nbest_fst = LAB / 'nbest.fst'
run_wsl('fstshortestpath', '--nshortest=8', '--unique=true',
        to_wsl_path(lattice_fst), to_wsl_path(nbest_fst))
openfst_nbest = parse_acceptor_paths(nbest_fst)
for rank, item in enumerate(openfst_nbest, start=1):
    print(f'{rank:2d}. {item["text"]:32s} cost={item["cost"]:.6f}')
assert [item['text'] for item in openfst_nbest] == [item['text'] for item in base_ranked]
assert all(abs(a['cost'] - b['total']) < 1e-5 for a, b in zip(openfst_nbest, base_ranked))


 1. jintian xinqing bucuo            cost=7.915209
 2. mingtian tianqi henhao           cost=7.942021
 3. zuotian tianqi bucuo             cost=8.136528
 4. jintian xinqing henhao           cost=8.215209
 5. jintian tianqi bucuo             cost=8.753529
 6. jintian tianqi hen leng          cost=8.816079
 7. jintian tianqi henhao            cost=9.353500
 8. jintian tianqi                   cost=9.356788


## 5. 交互观察 LM scale、插词项和热词

拖动参数前先预测第一名会怎样变化。热词奖励只降低包含该词的候选代价；它不应该直接篡改字符串，也不保证热词一定正确。


In [6]:
lm_slider = widgets.FloatSlider(value=1.0, min=0.0, max=2.0, step=0.1, description='LM scale')
insertion_slider = widgets.FloatSlider(value=0.0, min=-1.0, max=1.0, step=0.1, description='插词项')
hotword_dropdown = widgets.Dropdown(options=['', 'henhao', 'bucuo', 'tianqi'], value='', description='热词')
hotword_slider = widgets.FloatSlider(value=0.0, min=0.0, max=3.0, step=0.1, description='热词奖励')
score_output = widgets.Output()

def refresh_scores(change=None):
    ranked = rank_candidates(
        lm_scale=lm_slider.value, insertion_penalty=insertion_slider.value,
        hotword=hotword_dropdown.value, hotword_bonus=hotword_slider.value,
    )
    with score_output:
        score_output.clear_output(wait=True)
        for rank, item in enumerate(ranked[:5], start=1):
            print(f'{rank}. {item["text"]:32s} {item["total"]:.3f}')
        fig, ax = plt.subplots(figsize=(8, 3.2))
        top = ranked[:5][::-1]
        ax.barh([item['text'] for item in top], [item['total'] for item in top])
        ax.set_xlabel('总代价（越低越好）')
        ax.set_title('当前 Top-5')
        plt.tight_layout()
        plt.show()

for control in [lm_slider, insertion_slider, hotword_dropdown, hotword_slider]:
    control.observe(refresh_scores, names='value')
display(widgets.VBox([lm_slider, insertion_slider, hotword_dropdown, hotword_slider]), score_output)
refresh_scores()


Output()

## 6. Oracle WER：先判断正确答案是否还在候选里

如果正确文本已经在 beam pruning 时被删除，任何只做 N-best 重排的二遍 LM 都救不回来。`oracle WER@k` 是 top-k 中相对参考文本最小的 WER，用来区分“候选召回问题”和“重排问题”。


In [7]:
def edit_distance(reference, hypothesis):
    rows, cols = len(reference) + 1, len(hypothesis) + 1
    table = [[0] * cols for _ in range(rows)]
    for i in range(rows): table[i][0] = i
    for j in range(cols): table[0][j] = j
    for i in range(1, rows):
        for j in range(1, cols):
            substitution = table[i-1][j-1] + (reference[i-1] != hypothesis[j-1])
            table[i][j] = min(table[i-1][j] + 1, table[i][j-1] + 1, substitution)
    return table[-1][-1]

def wer(reference, hypothesis):
    ref_words, hyp_words = reference.split(), hypothesis.split()
    return edit_distance(ref_words, hyp_words) / max(1, len(ref_words))

REFERENCE = 'jintian tianqi henhao'
for k in [1, 3, 5, 8]:
    oracle = min(wer(REFERENCE, item['text']) for item in base_ranked[:k])
    print(f'oracle WER@{k} = {oracle:.3f}')
print('正确候选排名：', next(i for i, item in enumerate(base_ranked, 1) if item['text'] == REFERENCE))
assert wer(REFERENCE, base_ranked[0]['text']) > 0
assert min(wer(REFERENCE, item['text']) for item in base_ranked) == 0


oracle WER@1 = 0.667
oracle WER@3 = 0.333
oracle WER@5 = 0.333
oracle WER@8 = 0.000
正确候选排名： 7


## 7. 二遍重打分：先做“只重排，不生成”

下面的二遍代价是为教学人工设定的，模拟一个更擅长长上下文的 neural LM。它不会产生 lattice 之外的新句子，因此不会凭空改掉数字、人名或否定词。

真实实验必须在 validation set 调权重，并与只用一遍分数的基线比较；不能根据 test set 答案手调。


In [8]:
SECOND_PASS_COST = {
    'jintian tianqi henhao': 0.0,
    'jintian tianqi bucuo': 0.8,
    'jintian xinqing henhao': 0.9,
    'jintian xinqing bucuo': 1.3,
    'mingtian tianqi henhao': 1.8,
    'zuotian tianqi bucuo': 2.0,
    'jintian tianqi': 1.4,
    'jintian tianqi hen leng': 1.1,
}

def second_pass_rescore(first_pass_ranked, scale):
    rescored = [
        {**item, 'second_pass': SECOND_PASS_COST[item['text']],
         'rescored_total': item['total'] + scale * SECOND_PASS_COST[item['text']]}
        for item in first_pass_ranked
    ]
    return sorted(rescored, key=lambda item: item['rescored_total'])

for scale in [0.0, 0.5, 1.0, 1.5, 2.0]:
    best = second_pass_rescore(base_ranked, scale)[0]
    print(f'二遍 scale={scale:3.1f} → {best["text"]:30s} total={best["rescored_total"]:.3f}')
assert second_pass_rescore(base_ranked, 1.5)[0]['text'] == REFERENCE
assert {item['text'] for item in second_pass_rescore(base_ranked, 1.5)} == {item['text'] for item in base_ranked}


二遍 scale=0.0 → jintian xinqing bucuo          total=7.915
二遍 scale=0.5 → jintian xinqing bucuo          total=8.565
二遍 scale=1.0 → jintian xinqing henhao         total=9.115
二遍 scale=1.5 → jintian tianqi henhao          total=9.353
二遍 scale=2.0 → jintian tianqi henhao          total=9.353


## 8. 剪枝失败实验

本例正确句在一遍排序中比较靠后。若只保留 top-5，二遍模型再强也看不到它；保留 top-8 才能通过重排恢复。这不代表永远应该保存 8 条，而是说明 beam、lattice density 与二遍收益必须一起评测。


In [9]:
for keep_k in [1, 3, 5, 8]:
    kept = base_ranked[:keep_k]
    best = second_pass_rescore(kept, 1.5)[0]
    print(f'只保留 top-{keep_k}: 二遍结果={best["text"]:30s} WER={wer(REFERENCE, best["text"]):.3f}')
assert second_pass_rescore(base_ranked[:5], 1.5)[0]['text'] != REFERENCE
assert second_pass_rescore(base_ranked[:8], 1.5)[0]['text'] == REFERENCE


只保留 top-1: 二遍结果=jintian xinqing bucuo          WER=0.667
只保留 top-3: 二遍结果=jintian xinqing bucuo          WER=0.667
只保留 top-5: 二遍结果=jintian xinqing henhao         WER=0.333
只保留 top-8: 二遍结果=jintian tianqi henhao          WER=0.000


## 9. 自动判题

先闭卷填写。每题尽量只写一个关键词或短语，达到 7/8 再进入综合项目。


In [10]:
questions = [
    '1. 在本课 cost 域中，总代价越高还是越低越好？',
    '2. α 控制哪一种分数的强度？',
    '3. 正的插词项更倾向长句还是短句？',
    '4. 正的热词奖励在代价公式中应加还是减？',
    '5. 哪种结构会共享候选的公共前后缀？',
    '6. top-k 中理论上最小的 WER 叫什么？',
    '7. 正确候选已被剪掉，只重排能否恢复？',
    '8. 更安全的第一步是只重排候选，还是直接生成任意新文本？',
]
for question in questions:
    print(question)

answers = ['', '', '', '', '', '', '', '']
expected = [
    {'低', '越低越好', 'lower'}, {'lm', '语言模型', 'lm代价'}, {'短句', '短'},
    {'减', '减去', 'subtract'}, {'lattice'}, {'oracle wer', 'oraclewer'},
    {'不能', '否', 'no'}, {'只重排', '重排'},
]

def normalize_answer(value):
    return str(value).strip().lower().replace(' ', '')

score = 0
for index, (answer, accepted) in enumerate(zip(answers, expected), start=1):
    correct = normalize_answer(answer) in {normalize_answer(item) for item in accepted}
    score += int(correct)
    print(('✅' if correct else '❌'), f'第 {index} 题')
print(f'得分：{score}/8；达到 7/8 再进入综合项目。')


1. 在本课 cost 域中，总代价越高还是越低越好？
2. α 控制哪一种分数的强度？
3. 正的插词项更倾向长句还是短句？
4. 正的热词奖励在代价公式中应加还是减？
5. 哪种结构会共享候选的公共前后缀？
6. top-k 中理论上最小的 WER 叫什么？
7. 正确候选已被剪掉，只重排能否恢复？
8. 更安全的第一步是只重排候选，还是直接生成任意新文本？
❌ 第 1 题
❌ 第 2 题
❌ 第 3 题
❌ 第 4 题
❌ 第 5 题
❌ 第 6 题
❌ 第 7 题
❌ 第 8 题
得分：0/8；达到 7/8 再进入综合项目。


## 10. 离场票

- [ ] 我能用一句话区分 1-best、N-best 与 lattice；
- [ ] 我能写出融合代价公式，并解释每个符号的方向；
- [ ] 我能用 `fstshortestpath --nshortest` 提取候选；
- [ ] 我能解释 oracle WER@k 为什么用于诊断候选召回；
- [ ] 我能证明二遍重排无法恢复已经被剪掉的候选；
- [ ] 我能说明“只重排”为什么比无约束生成更适合作为第一个 LLM 基线；
- [ ] 自动判题至少 7/8。

下一课：[综合项目与闭卷验收](语言模型零基础_08_综合项目与闭卷验收.ipynb)：贯通开发集 PPL、N-best、融合调参、测试 WER、OpenFst 复核与可复现实验报告。
